# Cross-Validation



## Change you Runtime to **"T4"**

In [ ]:
#hide
#Run once per session
!pip install fastai wwf -q --upgrade

In [ ]:
#hide_input
from wwf.utils import state_versions
state_versions(['fastai', 'fastcore', 'wwf'])


---
This article is also a Jupyter Notebook available to be run from the top down. There
will be code snippets that you can then run in any environment.

Below are the versions of `fastai`, `fastcore`, and `wwf` currently running at the time of writing this:
* `fastai` : 2.8.7 
* `fastcore` : 1.12.26 
* `wwf` : 0.0.16 
---

## What is K-Fold Cross Validation?

* A way to get the most out of your data
* More models
* Ensembling
* Requires more training

## What is needed?

* Training set
* Test set

* Why no validation?

## Importing the Library

We will be doing a vision task so we'll import the vision library

In [ ]:
from fastai.vision.all import *

Below you will find the exact imports for everything we use today

In [ ]:
from fastcore.foundation import L

from fastai.callback.fp16 import to_fp16
from fastai.callback.progress import ProgressCallback
from fastai.callback.schedule import fit_one_cycle

from fastai.data.core import Datasets, show_at
from fastai.data.external import untar_data, URLs
from fastai.data.transforms import IntToFloatTensor, Normalize, ToTensor, IndexSplitter, get_image_files, parent_label, Categorize

from fastai.metrics import accuracy

from fastai.vision.augment import aug_transforms, RandomResizedCrop
from fastai.vision.core import PILImage, imagenet_stats
from fastai.vision.learner import cnn_learner

import random

from sklearn.model_selection import StratifiedKFold

from torchvision.models.resnet import resnet34

## ImageWoof

In [ ]:
path = untar_data(URLs.IMAGEWOOF)

In [ ]:
path.ls()

Scenario:

* We have a training set
* We have a test set

In [ ]:
item_tfms = [ToTensor(), RandomResizedCrop(460, min_scale=0.75, ratio=(1.,1.))]
batch_tfms = [IntToFloatTensor(), *aug_transforms(size=224, max_warp=0), Normalize.from_stats(*imagenet_stats)]
bs=64

We'll use the `IndexSplitter` just to get to know it. What we really wind up doing is a `RandomSplitter` split 80/20.

We can see `IndexSplitter`'s source code by doing:

In [ ]:
IndexSplitter??

Next let's get our images

In [ ]:
train_imgs = get_image_files(path/'train')
tst_imgs = get_image_files(path/'val')

We'll shuffle up our training set so the chance of including all classes is almost guarenteed

In [ ]:
random.shuffle(train_imgs)

In [ ]:
len(train_imgs)

And then we will do the 80/20 split

In [ ]:
train_imgs

In [ ]:
start_val = len(train_imgs) - int(len(train_imgs)*.2)
idxs = list(range(start_val, len(train_imgs)))
splitter = IndexSplitter(idxs)
splits = splitter(train_imgs)

Since we want to include our test set in with these splits, we'll make a `split_list` of all three of our splits (train, valid, test)

In [ ]:
split_list = [splits[0], splits[1]]

And we'll add in the range for our test set here:

In [ ]:
split_list.append(L(range(len(train_imgs), len(train_imgs)+len(tst_imgs))))

In [ ]:
split_list

Let's check that everything worked as intended. First building the `Datasets`:

In [ ]:
dsrc = Datasets(train_imgs+tst_imgs, tfms=[[PILImage.create], [parent_label, Categorize]],
                splits = split_list)

We can look at an item:

In [ ]:
show_at(dsrc.train, 3)

And if we check `n_subsets`, we can see that three are there (for our three splits)

In [ ]:
dsrc.n_subsets

3

Now let's build some `DataLoaders`

In [ ]:
dls = dsrc.dataloaders(bs=bs, after_item=item_tfms, after_batch=batch_tfms)

In [ ]:
dls.show_batch()

We can see the subsets was passed down to here as well:

In [ ]:
dls.n_subsets

What this means is while `dls.train` and `dls.valid` will return what we would expect, if we were to instead _index_ into our `DataLoader`, we can find our testing data in there too:

In [ ]:
dls[2].show_batch()

Let's do a quick baseline

In [ ]:
learn = cnn_learner(dls, resnet34, pretrained=False, metrics=accuracy).to_fp16()

In [ ]:
learn.fit_one_cycle(1)

Now how do we check it?

We can run `learn.validate` on our subset

In [ ]:
learn.validate(ds_idx=2)

## Now **how** do I do Cross-Validation?

#Stratified K-Fold

###Stratified K-Fold is a variation of K-Fold cross-validation. In standard K-Fold, the dataset is simply divided into K folds. However, with imbalanced datasets (where some classes are much rarer than others), a simple K-Fold split might result in some folds having very few or none of the minority class samples, leading to biased model evaluation.

###Stratified K-Fold addresses this by ensuring that each fold maintains the same proportion of observations for each target class as the complete dataset. For example, if you have 10% of your data belonging to class A, then each fold created by Stratified K-Fold will also have approximately 10% of its data from class A.

###This is particularly important for classification tasks, as it helps to ensure that each fold is a representative sample of the entire dataset, leading to more reliable and less biased performance estimates for your machine learning models.

First let's import our KFold

In [ ]:
from sklearn.model_selection import StratifiedKFold

And grab all the labels from our dataset

In [ ]:
train_labels = L(dsrc.items).map(dsrc.tfms[1])

Now let's make our K-Fold

In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True)

Finally we need to define a training loop to go over all our folds and gather our validation and test accuracy

In [ ]:
n_splits = 10

In [ ]:
import random
random.shuffle(train_imgs)

What's our loop going to look like?

In [ ]:
val_pct = []
tst_preds = []
skf = StratifiedKFold(n_splits=10, shuffle=True)
for _, val_idx in kf.split(np.array(train_imgs+tst_imgs), train_labels):
  splits = IndexSplitter(val_idx)
  split = splits(train_imgs)
  split_list = [split[0], split[1]]
  split_list.append(L(range(len(train_imgs), len(train_imgs)+len(tst_imgs))))
  dsrc = Datasets(train_imgs+tst_imgs, tfms=[[PILImage.create], [parent_label, Categorize]],
                  splits=split_list)
  dls = dsrc.dataloaders(bs=bs, after_item=item_tfms, after_batch=batch_tfms)
  learn = cnn_learner(dls, resnet34, pretrained=False, metrics=accuracy)
  learn.fit_one_cycle(1)
  val_pct.append(learn.validate()[1])
  a,b = learn.get_preds(ds_idx=2)
  tst_preds.append(a)

Now how do we combine all our predictions? We sum them all together then divide by our total (a voting ensemble is what this is referred to as)

First let's check the accuracy of one fold:

In [ ]:
tst_preds_copy = tst_preds.copy()
accuracy(tst_preds_copy[0], b)

Then we can print out all the folds. We can see our highest accuracy on the test set was 26.27%

In [ ]:
for i in tst_preds_copy:
  print(accuracy(i, b))

Now let's perform our vote:

In [ ]:
hat = tst_preds[0]
for pred in tst_preds[1:]:
  hat += pred

In [ ]:
hat

In [ ]:
hat /= len(tst_preds)

And see what our new accuracy is

In [ ]:
accuracy(hat, b)

That's an improvement ~2.5% or so! Not bad!

Ensembling in this way can have a diminishing return, so finding the right number of folds to use is something you should try to figure out through trial and error on subsamples of your dataset first.

#Leave-One-Out Cross-Validation (LOOCV)

###Leave-One-Out Cross-Validation (LOOCV) is an extreme case of K-Fold Cross-Validation where the number of folds K is equal to the number of data points in the dataset. In each iteration, one data point is used as the validation set, and the remaining N-1 data points are used as the training set. This process is repeated N times, ensuring that each data point is used exactly once as the validation set.

###While LOOCV provides a nearly unbiased estimate of the test error, it can be computationally very expensive, especially for large datasets, because it requires training the model N times. For this reason, it's generally not used for very large datasets.

###Let's implement a simplified version of LOOCV using sklearn.model_selection.LeaveOneOut on a subset of the train_imgs to demonstrate how it works. Since LeaveOneOut does not stratify, and for practical demonstration without excessive computation, we will use a small sample.

In [ ]:
from sklearn.model_selection import LeaveOneOut
import numpy as np

# For demonstration, let's take a small sample of the training images and their labels
# LOOCV is computationally intensive, so we'll use a very small subset.

sample_size = 500 # Adjust this to a larger number if you want a more representative, but slower, demo
sample_indices = np.random.choice(len(train_imgs), sample_size, replace=False)
sampled_train_imgs = L(train_imgs)[sample_indices]
sampled_train_labels = L(train_labels)[sample_indices]

print(f"Demonstrating LOOCV with {sample_size} samples.")

loo = LeaveOneOut()

# Loop through each split generated by LeaveOneOut
for i, (train_idx, val_idx) in enumerate(loo.split(sampled_train_imgs)):
    print(f"\nFold {i+1}:")
    print(f"  Training indices: {train_idx}")
    print(f"  Validation index: {val_idx}")

    # In a real scenario, you would create your Dataloaders and train a model here
    # using sampled_train_imgs[train_idx] for training and sampled_train_imgs[val_idx] for validation.

    # Example: Accessing the actual data for this fold
    train_data_for_fold = sampled_train_imgs[train_idx]
    val_data_for_fold = sampled_train_imgs[val_idx]

    print(f"  Training data count: {len(train_data_for_fold)}")
    print(f"  Validation data count: {len(val_data_for_fold)}")
    print(f"  Validation item: {val_data_for_fold[0].name}")


The output you see is a classic log of **Leave-One-Out Cross-Validation (LOOCV)**. To help you understand what is happening behind the scenes, we can break it down into these four key takeaways.

---

### 1. What is the "1" in Validation Data Count?
In standard K-Fold cross-validation, we split data into large chunks (e.g., 20% for testing). In **LOOCV**, the "Fold" size is exactly **one**.
* **The Logic:** If you have $N$ items in your dataset, you perform $N$ separate experiments.
* **In this output:** Notice how `Validation data count` is always **1**. The model is trained on 499 items and tested on the single remaining item (the "left-out" one).



### 2. The "Shifting Window" of Indices
Look closely at the `Validation index` and `Training indices` across Fold 360, 361, and 362:
* **Fold 360:** Validation index is `[359]`.
* **Fold 361:** Validation index is `[360]`.
* **Fold 362:** Validation index is `[361]`.

The validation index increments by one in every fold. Consequently, that specific index is **removed** from the `Training indices` list for that fold. For example, in Fold 361, the list of training indices skips from `359` directly to `361`.

### 3. Exhaustive Testing
The filenames (e.g., `n02088364_114.JPEG`) represent individual data points (in this case, images).
* By the time the code finishes, **every single image** in the dataset will have been used as the "test set" exactly once.
* This makes LOOCV **deterministic**: unlike K-Fold (which depends on how you randomly shuffle), LOOCV will always produce the exact same result for the same dataset.

### 4. Pros and Cons for the you to Note
Based on this specific output (showing 500 total samples), students should discuss:
* **Computational Cost:** To get a final accuracy score, the computer has to train the model **500 times**. If one training cycle takes 1 second, this takes ~8 minutes. If it were a larger dataset (e.g., 50,000 images), LOOCV would be nearly impossible.
* **Bias vs. Variance:** Because the training sets in each fold are 99.8% identical, the models are highly correlated. This leads to low bias but can result in high variance in the final error estimate.

---

### Suggested  Activity
**The "Skip the Jump" Challenge:**
Look at the `Training indices` block for **Fold 365**.
1. Find where the number `364` is missing.
2.  Find where the number `365` is present.
3.  *"If the validation index is [364], why is 365 included in the training set but 364 is not?"*

Answer: Because only the current validation point is excluded; points 'ahead' in the dataset are still fair game for training.

## Hold-out Cross-Validation

Hold-out cross-validation is the simplest form of cross-validation. The dataset is randomly divided into two subsets: a training set and a validation (or test) set. The model is trained exclusively on the training set and then evaluated on the validation set. This provides a single estimate of the model's performance.

**Advantages:**
*   Computationally efficient, as the model is trained only once.
*   Simple to understand and implement.

**Disadvantages:**
*   The performance estimate can be highly dependent on the specific division of the data, especially with smaller datasets.
*   A significant portion of data might not be used for training, potentially leading to a suboptimal model if the dataset is small.

In [ ]:
from fastai.data.transforms import RandomSplitter

# We'll use the 'train_imgs' for this demonstration of a simple hold-out split
# The test set (tst_imgs) is typically reserved for final, unseen evaluation.

# Define the splitter to randomly split 80% for training and 20% for validation
splitter_holdout = RandomSplitter(valid_pct=0.2, seed=42) # Using a seed for reproducibility
splits_holdout = splitter_holdout(train_imgs)

print(f"Number of training images: {len(splits_holdout[0])}")
print(f"Number of validation images: {len(splits_holdout[1])}")

In [ ]:
# Create Datasets and DataLoaders for the hold-out split
dsrc_holdout = Datasets(train_imgs, tfms=[[PILImage.create], [parent_label, Categorize]],
                        splits = splits_holdout)
dls_holdout = dsrc_holdout.dataloaders(bs=bs, after_item=item_tfms, after_batch=batch_tfms)

print(f"Number of subsets in DataLoaders: {dls_holdout.n_subsets}")
dls_holdout.show_batch()

In [ ]:
# Train a model using the hold-out sets
learn_holdout = cnn_learner(dls_holdout, resnet34, pretrained=False, metrics=accuracy).to_fp16()

print("Training model with hold-out validation...")
learn_holdout.fit_one_cycle(2) # Train for 2 epochs to see some progress

In [ ]:
# Evaluate the model on the hold-out validation set
print("Evaluating model on hold-out validation set...")
loss, acc = learn_holdout.validate()

print(f"Hold-out Validation Loss: {loss:.4f}")
print(f"Hold-out Validation Accuracy: {acc:.4f}")

Let us use the "Exam Analogy."

In machine learning, if you let a student (the model) practice on the exact same questions that will be on the final exam, they haven't learned—they've just memorized. Hold-out validation prevents this "cheating" (overfitting).



---

### 1. The Core Concept: "The Hidden Vault"
Hold-out validation is the simplest form of model evaluation. You take your entire dataset and split it into two unequal parts:
* **Training Set (usually 70–80%):** The "Textbook." The model uses this data to learn patterns, weights, and features.
* **Validation/Test Set (usually 20–30%):** The "Final Exam." This data is kept in a "hidden vault." The model **never** sees it during training. We only show it to the model at the very end to see how it handles brand-new, unseen information.

### 2. Interpreting Your Specific Results
Based on the output you provided:
> **Hold-out Validation Loss: 2.0390** > **Hold-out Validation Accuracy: 0.2753**

* **Accuracy (27.53%):** This is the "Score" on the final exam. It means the model correctly predicted the outcome for about 27 out of every 100 items it had never seen before.
* **Loss (2.0390):** This is the "Penalty" for being wrong. In classification, the higher this number, the more "confused" or "confident in its wrongness" the model was.
* **The Verdict:** An accuracy of 27% is generally quite low (depending on the number of classes). It suggests the model is either **underfitting** (too simple to learn) or the patterns in the training set didn't translate well to the hold-out set.

---

### 3. Pros & Cons

| Feature | Why it's good/bad |
| :--- | :--- |
| **Speed** | Very fast! You only train the model **once**. |
| **Simplicity** | Easy to understand and implement with `train_test_split`. |
| **The "Luck" Factor** | **Risk:** If you get a "lucky" split where the hold-out set is very easy, your accuracy will look better than it actually is. |
| **Data Waste** | Since 20–30% of data is "held out," the model has fewer examples to learn from. |